# 🛰️ GeoNexa — RemoteCLIP Foundation Model Fine-Tuning
### Production-Grade Remote Sensing Image-Text Contrastive Learning (SIH 2026)

This notebook trains and fine-tunes **RemoteCLIP (ViT-B-32)** on Earth Observation satellite imagery and real remote-sensing image-text pairs using a GPU (T4 / A100 / V100).

---
### Workflow Overview:
1. **Environment Setup**: Clone repository & install dependencies
2. **Download Weights & Data**: Download base RemoteCLIP weights (`RemoteCLIP-ViT-B-32.pt`) and generate training pairs
3. **GPU Contrastive Fine-Tuning**: Train with InfoNCE loss + Cosine LR Schedule + AdamW on GPU (15-20 epochs)
4. **WiSE-FT Weight Interpolation & Export**: Export deployment-ready model and auto-download to your local PC

## 1. Environment & GPU Verification

In [ ]:
!nvidia-smi
!git clone https://github.com/Amityush-lgtm/GeoNexa.git
%cd GeoNexa
!pip install -q open-clip-torch rasterio scipy pillow torchvision tqdm gdown faiss-cpu

## 2. Download Pretrained RemoteCLIP Foundation Weights

In [ ]:
!python scripts/download_models.py --force

## 3. Prepare Multi-Spectral Training Data & Captions

In [ ]:
# Generate Sentinel-2 scenes & spectral tiles
!python scripts/stage_guwahati_sentinel2.py
!python scripts/ingest.py --data-dir data/public/raw/sentinel2

# Extract multi-spectral indices (NDVI, NDWI, NDBI) and generate training pairs
!python scripts/prepare_training_data.py --tiles-dir data/public/tiles --output training_data/ --captions-per-tile 16

## 4. Run GPU Contrastive Fine-Tuning

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on device: {device}')

# Run 15-epoch fine-tuning on GPU with batch size 16 (or 32 on A100)
!python -m training.train \
    --train-data training_data/train_pairs.json \
    --val-data training_data/val_pairs.json \
    --checkpoint models/remoteclip-vit-b-32/RemoteCLIP-ViT-B-32.pt \
    --output-dir checkpoints \
    --epochs 15 \
    --batch-size 16 \
    --lr 1e-5 \
    --device cuda \
    --unfreeze-last-n 4 \
    --eval-every 1 \
    --save-every 1 \
    --experiment-name geonexa_gpu_production

## 5. Export Fine-Tuned Model with WiSE-FT Weight Interpolation & Download

In [ ]:
# Export with WiSE-FT (Weight-Space Ensemble) blending fine-tuned + base zero-shot representations
!python scripts/export_finetuned_model.py \
    --checkpoint checkpoints/best_model.pt \
    --base-model models/remoteclip-vit-b-32/RemoteCLIP-ViT-B-32.pt \
    --alpha 0.4 \
    --output models/RemoteCLIP-ViT-B-32-finetuned.pt \
    --verify

# Download the exported model weights directly to your local PC
from google.colab import files
files.download('models/RemoteCLIP-ViT-B-32-finetuned.pt')
files.download('checkpoints/training_report.json')

## 6. How to Deploy to Your Local GeoNexa Backend

Once downloaded to your PC:
1. Move `RemoteCLIP-ViT-B-32-finetuned.pt` into `d:\SIH 2.0\semantic-eo-search\models\`
2. Rebuild the FAISS index:
   ```bash
   python scripts/build_index.py --model-path models/RemoteCLIP-ViT-B-32-finetuned.pt --force-rebuild
   ```
3. Run retrieval evaluation:
   ```bash
   python scripts/evaluate_retrieval.py
   ```